# GenAI, LangChain, RAG, Agents, Memory and Optimization

## Purpose of this notebook

This notebook explains each concept in **deeper, process-oriented detail**:

- What the concept is
- Why it is needed
- How the process works step by step
- Where it is used
- What alternatives exist
- Why one approach may be chosen over another
- How to optimize latency, token count, API calls, cost, and accuracy
- Interview-ready explanations

The goal is not only to know definitions, but to understand the **complete flow of a production GenAI application**.

# 1. What is Generative AI?

Generative AI is a type of artificial intelligence that creates new content such as text, code, images, audio, or summaries.

A Large Language Model (LLM) does not directly "understand" text like a human. It processes input as **tokens** and predicts the most probable next token based on patterns learned during training.

## Basic generation process

1. User sends a prompt.
2. The tokenizer converts the prompt into tokens.
3. The tokens are converted into numerical representations.
4. The transformer model processes the relationships between tokens.
5. The model predicts probabilities for the next token.
6. A decoding strategy selects one token.
7. The selected token is added to the sequence.
8. The process repeats until the response is complete.

## Why plain LLMs are not enough

A standalone LLM has important limitations:

- It may not know private company data.
- Its training knowledge may be outdated.
- It can hallucinate.
- It cannot automatically access databases or APIs.
- It has a limited context window.
- It does not naturally remember previous conversations permanently.

Frameworks such as LangChain and LangGraph help build applications around the LLM to solve these problems.

# 2. LangChain, LangGraph and LangSmith

## LangChain

LangChain is a framework for building LLM applications by connecting models with prompts, retrievers, vector databases, tools, memory, and output parsers.

### Process

**User Input → Prompt → LLM / Retriever / Tool → Processing → Final Output**

LangChain provides reusable components so developers do not have to manually write all orchestration logic.

### Where it is useful

- RAG applications
- Chatbots
- Document question answering
- Tool calling
- API integration
- Agents
- Structured output generation

---

## LangGraph

LangGraph is used for **stateful, multi-step, controllable agent workflows**.

A simple chain usually moves forward in a fixed sequence. LangGraph represents the application as a graph:

- **Nodes** = tasks or functions
- **Edges** = movement between tasks
- **State** = information shared across the workflow
- **Conditional edges** = decisions about what happens next
- **Cycles** = ability to repeat a step

### Example process

User Question  
↓  
Agent analyzes question  
↓  
Does it need external information?

- No → Answer directly
- Yes → Select tool → Execute tool → Observe result → Decide again

This loop can continue until the task is complete.

### Why use LangGraph instead of a basic chain?

Use LangGraph when:

- The workflow contains decisions.
- The agent may call tools multiple times.
- Steps may repeat.
- State must be maintained.
- Human approval is required.
- The workflow needs better control and debugging.

---

## LangSmith

LangSmith is used for observability, tracing, debugging, evaluation, and monitoring of LLM applications.

### Process

Application runs  
↓  
LangSmith captures traces  
↓  
Developer inspects:

- Prompt
- Retrieved documents
- Model input
- Model output
- Tool calls
- Latency
- Errors
- Token usage

### Why it matters

Without tracing, a wrong answer only tells us that the application failed. LangSmith helps identify **where** it failed:

- Retrieval problem?
- Bad prompt?
- Wrong tool?
- Weak model?
- Too much context?
- Output parsing failure?

# 3. What does "Chain" mean in LangChain?

A chain is a sequence of connected processing steps where the output of one step becomes the input of another.

## Simple example

**User Question → Prompt Template → LLM → Output Parser → Final Answer**

### Detailed process

1. Receive the user question.
2. Insert the question into a prompt template.
3. Send the completed prompt to the LLM.
4. Receive the raw model output.
5. Parse or structure the output.
6. Return the result.

## RAG chain example

**Question → Retriever → Documents → Prompt → LLM → Answer**

A chain is best when the workflow is predictable.

## Chain vs Agent

| Chain | Agent |
|---|---|
| Fixed workflow | Dynamic workflow |
| Developer defines steps | LLM decides next action |
| Predictable | Flexible |
| Usually fewer API calls | May use many API calls |
| Lower latency | Usually higher latency |
| Easier to test | More complex to test |

### Decision rule

If the steps are already known, use a **chain**.  
If the application must decide what action to take, use an **agent**.

# 4. What is RAG?

RAG means **Retrieval-Augmented Generation**.

It combines:

1. **Retrieval** — find relevant information from external knowledge.
2. **Generation** — give that information to an LLM to generate an answer.

## Why RAG is needed

Suppose an employee asks:

> What is our company's leave policy?

A public LLM may not know the private company policy.

RAG solves this by retrieving the correct section from company documents and giving it to the LLM.

## Complete RAG process

### Offline phase: Indexing

Documents  
↓  
Load documents  
↓  
Clean text  
↓  
Split into chunks  
↓  
Create embeddings  
↓  
Store vectors + text + metadata in vector database

### Online phase: Query and answer

User Question  
↓  
Create query embedding  
↓  
Search vector database  
↓  
Retrieve relevant chunks  
↓  
Optional filtering / reranking  
↓  
Build prompt with context  
↓  
Send prompt to LLM  
↓  
Generate grounded answer

## Important point

The vector database usually stores:

- Chunk text
- Embedding vector
- Metadata

Example metadata:

- File name
- Page number
- Department
- Date
- Document type

# 5. Types of RAG

## 1. Naive / Basic RAG

**Query → Retrieve → Generate**

Best for simple document Q&A.

Problem: retrieval quality may be weak.

---

## 2. Advanced RAG

Adds improvements before and after retrieval.

**Query → Query Optimization → Retrieval → Reranking → Context Compression → LLM**

Techniques:

- Query rewriting
- Metadata filtering
- Hybrid search
- Reranking
- Context compression

Best for production systems requiring better accuracy.

---

## 3. Modular RAG

Different components are independently designed and combined.

Example:

**Router → SQL Retriever / Vector Retriever / Web Search → Reranker → LLM**

Best when multiple data sources are involved.

---

## 4. Agentic RAG

An agent decides:

- Whether retrieval is needed
- Which source to search
- Whether the result is sufficient
- Whether another search is required

Process:

Question  
↓  
Agent plans  
↓  
Select source  
↓  
Retrieve  
↓  
Evaluate result  
↓  
Enough information?

- Yes → Generate answer
- No → Search again

Best for complex research and multi-source questions.

---

## 5. Graph RAG

Information is represented using entities and relationships.

Example:

**Employee → Works On → Project → Uses → Technology**

Best for questions involving relationships, dependencies, and multi-hop reasoning.

# 6. Chunking

Chunking means splitting a large document into smaller pieces before creating embeddings.

## Why chunking is necessary

Embedding an entire large document as one vector causes problems:

- Different topics get mixed.
- Retrieval becomes less precise.
- Too much irrelevant text reaches the LLM.
- Token usage increases.

## Chunking process

Document  
↓  
Clean text  
↓  
Choose chunking strategy  
↓  
Split text  
↓  
Add overlap if needed  
↓  
Attach metadata  
↓  
Create embedding for each chunk  
↓  
Store in vector database

## Major chunking techniques

### 1. Fixed-size chunking

Split by a fixed number of characters or tokens.

Example: 500 tokens with 50-token overlap.

**Advantages:** Simple and fast.  
**Disadvantages:** Can cut sentences or ideas in the middle.

---

### 2. Recursive chunking

Try larger natural separators first:

1. Paragraph
2. New line
3. Sentence
4. Word

This is a common default because it preserves structure better than fixed splitting.

---

### 3. Sentence-based chunking

Groups complete sentences.

Best for:

- Articles
- Policies
- FAQs

Problem: sentence lengths vary.

---

### 4. Semantic chunking

Uses embeddings to detect topic changes.

Process:

1. Split into sentences.
2. Create sentence embeddings.
3. Compare neighboring sentence similarity.
4. Start a new chunk when semantic similarity drops.

Best for long documents containing multiple topics.

Trade-off: more embedding calls and higher indexing cost.

---

### 5. Structure-aware chunking

Uses document structure:

- Headings
- Sections
- Tables
- Markdown
- HTML tags
- Code functions/classes

Best for structured documents and technical documentation.

## How to choose chunk size

Small chunks:

- Better precision
- More vectors
- More storage
- May lose context

Large chunks:

- Better context
- Fewer vectors
- More irrelevant information
- More prompt tokens

The correct size must be tested using retrieval evaluation, not guessed.

# 7. Embeddings and Vector Storage

An embedding converts text into a numerical vector that represents semantic meaning.

Example conceptually:

> "car" → [0.12, -0.44, 0.81, ...]

Similar meanings produce vectors that are closer in vector space.

## Data storage process

### Step 1: Load document

Example: PDF, Word file, website, database row.

### Step 2: Extract and clean text

Remove unnecessary headers, repeated footers, broken characters, etc.

### Step 3: Chunk the text

Split the document into meaningful pieces.

### Step 4: Generate embeddings

Each chunk is sent to an embedding model.

### Step 5: Store data

Store:

- Vector
- Original chunk
- Metadata

### Step 6: Query

Convert the user query into an embedding.

### Step 7: Similarity search

Compare the query vector with stored vectors.

Common similarity measures:

- Cosine similarity
- Dot product
- Euclidean distance

### Step 8: Retrieve Top-K chunks

Return the most similar chunks to the application.

# 8. What is a Vector Database?

A Vector Database (VDB) stores and searches high-dimensional embeddings efficiently.

A normal relational database is optimized for exact matching:

`employee_id = 101`

A vector database is optimized for semantic similarity:

`Find text that means something similar to this question.`

## FAISS vs ChromaDB vs Pinecone

| Feature | FAISS | ChromaDB | Pinecone |
|---|---|---|---|
| Type | Similarity search library | Vector database | Managed cloud vector DB |
| Hosting | Local | Local / server | Cloud managed |
| Setup | Developer manages | Easy | Very easy operationally |
| Scaling | Manual | Moderate | Strong production scaling |
| Metadata filtering | Limited compared with DBs | Supported | Strong |
| Best use | Experiments, local search | Prototypes and smaller apps | Production cloud applications |

## Selection logic

Choose **FAISS** when:

- You want local experimentation.
- You need fast similarity search.
- You do not need a full managed database.

Choose **ChromaDB** when:

- You want easy local RAG development.
- You need document + metadata storage.
- You are building a prototype.

Choose **Pinecone** when:

- You need managed infrastructure.
- You expect production scale.
- You need strong filtering and operational simplicity.

The choice is not about which is universally "best." It depends on scale, cost, filtering, deployment, and operational requirements.

# 9. Query Decomposition

Query decomposition means splitting a complex question into smaller sub-questions.

## Example

User asks:

> Compare 2025 sales of Product A and Product B and explain which region caused the difference.

This contains multiple tasks:

1. Find Product A sales.
2. Find Product B sales.
3. Compare them.
4. Find regional performance.
5. Explain the reason for the difference.

## Process

Complex Query  
↓  
LLM decomposes query  
↓  
Generate sub-queries  
↓  
Retrieve information for each sub-query  
↓  
Combine evidence  
↓  
Generate final answer

## Why use it?

A single embedding may not represent all parts of a complex question well. Decomposition improves multi-step retrieval.

## Trade-offs

Advantages:

- Better coverage
- Better multi-hop reasoning

Disadvantages:

- More retrieval calls
- More LLM calls
- Higher latency
- Higher cost

## Optimization

Do not decompose every query. First classify query complexity.

Simple query → direct retrieval  
Complex query → decomposition

# 10. Reranking

Initial vector search is fast, but the highest similarity score does not always mean the chunk is the most useful.

Reranking reorders retrieved documents using a stronger relevance model.

## Process

Query  
↓  
Vector search retrieves Top 20 chunks  
↓  
Reranker scores query + each chunk more accurately  
↓  
Keep best 3–5 chunks  
↓  
Send only those chunks to LLM

## Why reranking improves RAG

Without reranking:

- Irrelevant chunks may enter the prompt.
- Token count increases.
- LLM may become confused.

With reranking:

- Better context quality
- Fewer prompt tokens
- Better answer accuracy

## Reranking techniques

### 1. Cross-encoder reranking

The model reads the query and document together and produces a relevance score.

High accuracy, but slower.

### 2. LLM-based reranking

An LLM evaluates relevance.

Flexible, but expensive and slow.

### 3. Reciprocal Rank Fusion (RRF)

Combines ranked results from multiple retrieval methods.

Useful for hybrid search.

### 4. Metadata-based reranking

Boost documents based on:

- Recency
- Department
- Document type
- User permissions

## Optimization

Bad approach:

Retrieve 100 chunks → rerank all 100 → send 20 to LLM

Better approach:

Retrieve 15–30 candidates → rerank → keep 3–8 strong chunks

This reduces reranking cost and final prompt tokens.

# 11. Tools in LangChain

A tool is a function that an LLM or agent can call to interact with an external system.

Examples:

- Search database
- Call REST API
- Search web
- Read file
- Execute calculation
- Get weather
- Create ticket

## Custom tool process

1. Developer creates a Python function.
2. Define a clear tool name.
3. Define the input schema.
4. Write a precise tool description.
5. Register the tool with the agent.
6. User asks a question.
7. LLM decides whether the tool is required.
8. LLM generates tool arguments.
9. Application executes the tool.
10. Tool result returns to the LLM.
11. LLM uses the result to answer.

## Example

User:

> What is the status of ticket INC00123?

Process:

Agent understands intent  
↓  
Selects `get_incident_status` tool  
↓  
Passes `INC00123`  
↓  
Tool calls ServiceNow API  
↓  
API returns status  
↓  
Agent explains result to user

## Why tools led to Agentic AI

A plain LLM can only generate text.

With tools, the system can:

- Retrieve live information
- Perform actions
- Interact with applications
- Make decisions based on observations

This changes the system from a text generator into an action-oriented agent.

In [ ]:
# Conceptual custom tool example

def get_order_status(order_id: str):
    # In a real application, this function would call a database or API.
    return {"order_id": order_id, "status": "Shipped"}

print(get_order_status("ORD-101"))

# 12. What is Memory?

Memory allows a conversational application to use information from previous interactions.

## Why memory is needed

Without memory:

User: My name is Ravi.  
User: What is my name?  
Bot: I do not know.

With memory, the application can use previous context.

## Types of memory

### 1. Full conversation buffer

Stores the complete conversation.

Best for short conversations.

Problem: token count grows continuously.

---

### 2. Window memory

Keeps only the most recent N messages.

Best for conversations where recent context matters most.

Advantage: predictable token usage.

Problem: older important facts disappear.

---

### 3. Summary memory

Older conversation is summarized.

Process:

Old messages  
↓  
Generate summary  
↓  
Store summary  
↓  
Combine summary + recent messages

Best for long conversations.

Trade-off: summarization requires additional model calls and may lose details.

---

### 4. Session memory

Stores context only during one user session.

Best for temporary support or task workflows.

When the session ends, memory may be cleared.

---

### 5. Long-term memory

Stores important facts across sessions.

Usually stored in:

- Database
- Key-value store
- Vector database

Best for personalization.

Problem: requires careful privacy, relevance, deletion, and update handling.

---

### 6. Semantic memory

Stores facts and retrieves them by meaning.

Example:

Stored: "User prefers vegetarian food."

Later query: "What restaurant should I recommend?"

The system retrieves the preference even if the wording is different.

# 13. Memory Optimization for Chatbots

Sending the full conversation on every request is a poor scaling strategy.

## Problem

Suppose conversation history contains 20,000 tokens.

Every new question may send those 20,000 tokens again.

Results:

- Higher cost
- Higher latency
- Context window pressure
- More irrelevant information

## Better architecture

Recent messages  
+  
Summary of older conversation  
+  
Only relevant long-term memories

## Process

New message arrives  
↓  
Keep recent conversation window  
↓  
Retrieve relevant long-term memories  
↓  
Use existing summary for older history  
↓  
Build limited context  
↓  
Call LLM  
↓  
Periodically update summary

## Optimization techniques

- Sliding window
- Summarization
- Semantic retrieval
- Store only important facts
- Delete duplicate memories
- Add expiration for temporary information
- Use smaller models for summarization
- Summarize periodically instead of every turn

The goal is not maximum memory. The goal is **minimum relevant memory required for a correct answer**.

# 14. What is an Agent?

An agent is an LLM-based system that can reason about a task, choose actions, use tools, observe results, and continue until it reaches a goal.

## Basic agent loop

User Goal  
↓  
Analyze task  
↓  
Choose action  
↓  
Call tool  
↓  
Observe result  
↓  
Is task complete?

- No → choose another action
- Yes → generate final response

## Example

User:

> Find my highest sales region and send me a summary.

Possible agent process:

1. Query sales database.
2. Analyze returned data.
3. Identify highest region.
4. Generate summary.
5. Call email tool.

## Agent vs fixed workflow

Use an agent when the path is unknown.

Do not use an agent when the process is always:

A → B → C

A fixed chain is usually faster, cheaper, and easier to control.

# 15. Complete Production RAG Process

## Offline indexing pipeline

Raw Documents  
↓  
Document Loader  
↓  
Text Cleaning  
↓  
Chunking  
↓  
Metadata Enrichment  
↓  
Embedding Model  
↓  
Vector Database

## Online query pipeline

User Query  
↓  
Query Classification  
↓  
Optional Query Rewriting / Decomposition  
↓  
Query Embedding  
↓  
Vector / Hybrid Search  
↓  
Metadata Filtering  
↓  
Reranking  
↓  
Context Compression  
↓  
Prompt Construction  
↓  
LLM  
↓  
Output Validation  
↓  
Final Answer

## Evaluation pipeline

Measure:

- Retrieval accuracy
- Answer correctness
- Faithfulness
- Latency
- Token usage
- API calls
- Cost

# 16. Latency, Tokens, API Calls and Cost Optimization

A production GenAI system should not only produce correct answers. It must also be fast and affordable.

## Main sources of latency

- LLM calls
- Embedding calls
- Vector database search
- Reranking
- Tool/API calls
- Large prompts
- Sequential workflows

## Token optimization

### Bad approach

Send:

- Entire document
- Entire conversation
- 20 retrieved chunks
- Repeated instructions

### Better approach

Send:

- Only relevant chunks
- Short system instructions
- Summary of old conversation
- Recent messages
- Compressed context

## API call optimization

Before adding an LLM call, ask:

> Can deterministic code do this?

Examples:

- Simple calculation → Python/code, not LLM
- Exact database lookup → direct query
- Fixed routing rule → code
- Complex natural-language classification → LLM may be useful

## Reduce sequential calls

Slow:

Call 1 → wait → Call 2 → wait → Call 3

Faster when independent:

Call 1 + Call 2 + Call 3 in parallel

## Cache repeated work

Cache:

- Embeddings
- Repeated queries
- Tool results
- Stable summaries

## Use the right model for the task

Use smaller/cheaper models for:

- Classification
- Query rewriting
- Summarization

Use stronger models only where complex reasoning is required.

# 17. How to Prove Optimization

Saying "the system is optimized" is not enough. Measure before and after.

## Metrics

### Latency

`Total response time = retrieval + reranking + LLM + tools`

Measure:

- Average latency
- P95 latency
- Time to first token

### Token count

Measure:

- Input tokens
- Output tokens
- Total tokens per request

### API calls

Count:

- LLM calls
- Embedding calls
- Tool calls
- Database calls

### Cost

`Cost per request × number of requests`

### Accuracy

Measure:

- Retrieval relevance
- Groundedness
- Answer correctness
- Hallucination rate

## Example comparison

| Metric | Before | After |
|---|---:|---:|
| Retrieved chunks sent to LLM | 15 | 5 |
| Input tokens | 9,000 | 3,200 |
| LLM calls | 4 | 2 |
| Average latency | 8 sec | 4 sec |
| Answer accuracy | 78% | 87% |

This is how an optimization claim should be demonstrated.

# 18. Why This Approach and Not Another?

In interviews and project explanations, always connect the technology choice to a measurable requirement.

## Example: Why RAG instead of fine-tuning?

Use RAG when:

- Knowledge changes frequently.
- Sources must be updated easily.
- Answers should be grounded in documents.
- Citations are needed.

Fine-tuning is better for changing behavior, style, or task patterns—not for continuously updating factual knowledge.

## Why reranking?

Because vector similarity alone may return semantically related but less useful chunks.

## Why summary memory?

Because full conversation history causes token growth.

## Why a chain instead of an agent?

Because a fixed workflow needs fewer decisions, fewer model calls, lower latency, and better predictability.

## Why an agent?

Because the application must dynamically choose tools and steps.

## Why Pinecone instead of FAISS?

Because managed scaling and filtering may matter more than local simplicity.

Every architecture decision should be explained using:

**Requirement → Options → Trade-off → Decision → Measured Result**

# 19. End-to-End Example: Retail Demand Forecasting Assistant

A user asks:

> Which products are at risk of stockout next week, and what should we restock?

## Process

1. User submits the question.
2. System classifies the question as inventory analysis.
3. Tool retrieves forecast results from the ML model/database.
4. Retriever searches relevant product and business context.
5. If the question is complex, it is decomposed:
   - Which products have high predicted demand?
   - Which products have low inventory?
   - Which products have the largest demand-supply gap?
6. Retrieved context is reranked.
7. Only the best context is sent to the LLM.
8. LLM generates a business-friendly explanation.
9. Output is validated against the retrieved data.
10. Final recommendation is returned.

## Why this architecture?

- ML model handles numerical forecasting.
- Database/tool provides exact live values.
- RAG provides business context.
- LLM explains results in natural language.

Using an LLM alone for all four tasks would be less reliable.

# 20. Interview Summary

## LangChain
Framework that connects LLMs with prompts, retrieval, tools, memory, and application logic.

## LangGraph
Graph-based orchestration for stateful workflows, decisions, loops, and agents.

## LangSmith
Tracing, debugging, evaluation, and monitoring.

## RAG
Retrieve relevant external knowledge and give it to the LLM before generation.

## Chunking
Split large documents into retrievable units.

## Embeddings
Numerical semantic representations of text.

## Vector Database
Stores embeddings and performs similarity search.

## Query Decomposition
Break complex questions into smaller retrievable questions.

## Reranking
Reorder retrieved chunks using a stronger relevance method.

## Tools
Functions that allow an LLM/agent to access external systems and perform actions.

## Memory
Stores and retrieves useful conversation context.

## Agent
Dynamically decides which actions and tools to use.

## Optimization
Reduce unnecessary tokens, API calls, context, retrieval candidates, and sequential steps while measuring accuracy and latency.